# Water Quality Prediction: XGBoost Regression

This Snowflake notebook follows the same data/feature workflow as the provided benchmark notebook, but swaps the model to the one shown in the title.

**Targets:** Total Alkalinity (TA), Electrical Conductance (EC), Dissolved Reactive Phosphorus (DRP)
**Features (same baseline set):** swir22, NDMI, MNDWI, pet
**Metrics:** R² and RMSE (Train + Test)

> Reminder: Do not use Latitude/Longitude directly as model features per challenge guidance.

## 1) Install dependencies (Snowflake container runtime)
If you already ran the benchmark/getting-started notebooks, you may have `requirements.txt` in your notebook files.
Run the cell below once per fresh container session, then restart the kernel if required.

In [ ]:
# If running on Snowflake container runtime, install packages from requirements.txt
# STRICT: try common paths without assuming folder structure
!pip install uv

import os
req_candidates = ['requirements.txt', './requirements.txt', '../requirements.txt']
req_path = None
for p in req_candidates:
    if os.path.exists(p):
        req_path = p
        break

if req_path is not None:
    !uv pip install -r "{req_path}"
else:
    print('requirements.txt not found in common locations; skipping requirements install')

# XGBoost may not be included in requirements.txt; install if needed
!uv pip install xgboost


## 2) Imports

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error

try:
    from IPython.display import display
except Exception:
    display = print
from xgboost import XGBRegressor

## 3) Load training data
This expects the same CSVs used by the benchmark notebook to be in the notebook Files panel:
- `water_quality_training_dataset.csv`
- `landsat_features_training.csv`
- `terraclimate_features_training.csv`

In [ ]:
# ============================================================
# STRICT Snowflake-friendly file loader
# Tries common relative locations without assuming OS root.
# ============================================================
from pathlib import Path

def read_csv_strict(filename: str) -> pd.DataFrame:
    candidates = [
        Path(filename),
        Path('./') / filename,
        Path('../') / filename,
    ]
    for p in candidates:
        try:
            if p.exists():
                return pd.read_csv(p.as_posix())
        except Exception:
            pass
    # Last attempt: let pandas raise a clear error
    return pd.read_csv(filename)


In [ ]:
Water_Quality_df = read_csv_strict('water_quality_training_dataset.csv')
landsat_train_features = read_csv_strict('landsat_features_training.csv')
Terraclimate_df = read_csv_strict('terraclimate_features_training.csv')

# Ensure NDMI/MNDWI are numeric
for col in ['NDMI', 'MNDWI']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = pd.to_numeric(landsat_train_features[col], errors='coerce')

display(Water_Quality_df.head())
display(landsat_train_features.head())
display(Terraclimate_df.head())


## 4) Join datasets + impute missing values
We concatenate columns (dropping duplicates) and median-impute numeric missing values (same strategy as benchmark).

In [ ]:
def combine_three_datasets(dataset1, dataset2, dataset3):
    data = pd.concat([dataset1, dataset2, dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_three_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
display(wq_data.head())

## 5) Select features + targets
We follow the baseline feature set used in the benchmark workflow: `swir22`, `NDMI`, `MNDWI`, `pet`.

In [ ]:
# ============================================================
# 5) Select features + targets
# Baseline feature set (benchmark-like) + Feature-training set (wider)
# ============================================================
BASE_FEATURE_COLS = ['swir22', 'NDMI', 'MNDWI', 'pet']
TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# STRICT: Do not use Latitude/Longitude directly as features
PROHIBITED_FEATURE_COLS = {'Latitude','Longitude','latitude','longitude','lat','lon','lng'}

# Build a wide numeric feature set from merged data (Landsat + TerraClimate + any other numeric cols)
numeric_cols = wq_data.select_dtypes(include=[np.number]).columns.tolist()
candidate_cols = [c for c in numeric_cols if c not in TARGET_COLS and c not in PROHIBITED_FEATURE_COLS]

# Ensure baseline features exist
missing_base = [c for c in BASE_FEATURE_COLS if c not in wq_data.columns]
if missing_base:
    raise ValueError(f'Missing baseline feature columns: {missing_base}')

# Dataframes
wq_model_df_base = wq_data[BASE_FEATURE_COLS + TARGET_COLS].copy()
X_base = wq_model_df_base[BASE_FEATURE_COLS].apply(pd.to_numeric, errors='coerce')

wq_model_df_full = wq_data[candidate_cols + TARGET_COLS].copy()
X_full = wq_model_df_full[candidate_cols].apply(pd.to_numeric, errors='coerce')

y_TA = wq_model_df_full['Total Alkalinity']
y_EC = wq_model_df_full['Electrical Conductance']
y_DRP = wq_model_df_full['Dissolved Reactive Phosphorus']

print('Baseline feature count:', X_base.shape[1])
print('Full numeric feature count:', X_full.shape[1])
display(wq_model_df_base.head())


## 6) Helper functions (split, scale, train, evaluate)
We compute Train and Test metrics for each target to spot overfitting.

In [ ]:
# ============================================================
# 6) Helper functions (split, impute, train, evaluate)
# NOTE: XGBoost (tree-based) does not require feature scaling.
# We focus on robust imputation + feature selection (feature training).
# ============================================================
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def impute_data(X_train, X_test):
    imputer = SimpleImputer(strategy='median')
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_imp  = pd.DataFrame(imputer.transform(X_test),  columns=X_test.columns,  index=X_test.index)
    return X_train_imp, X_test_imp, imputer

def train_xgb(X_train, y_train, X_eval, y_eval, random_state=42):
    # Stronger starter params + early stopping
    model = XGBRegressor(
        n_estimators=3000,
        learning_rate=0.02,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        min_child_weight=1.0,
        objective='reg:squarederror',
        random_state=random_state,
        n_jobs=-1
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_eval, y_eval)],
        verbose=False,
        early_stopping_rounds=100
    )
    return model

MODEL_NAME = 'XGBRegressor'
def evaluate_model(model, X_df, y_true, dataset_name='Test'):
    y_pred = model.predict(X_df)
    r2 = float(r2_score(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    print(f'\n{dataset_name} Evaluation:')
    print(f'R²: {r2:.4f}')
    print(f'RMSE: {rmse:.4f}')
    return y_pred, r2, rmse

def run_pipeline_feature_training(X, y, param_name='Parameter', top_n=80):
    print('\n' + '='*60)
    print(f'Feature-training XGBRegressor for {param_name}')
    print('='*60)

    # Split
    X_train, X_test, y_train, y_test = split_data(X, y)

    # Impute
    X_train_imp, X_test_imp, imputer = impute_data(X_train, X_test)

    # 1) First model on ALL features
    model_all = train_xgb(X_train_imp, y_train, X_test_imp, y_test)
    _, r2_train_all, rmse_train_all = evaluate_model(model_all, X_train_imp, y_train, 'Train (ALL)')
    _, r2_test_all, rmse_test_all = evaluate_model(model_all, X_test_imp, y_test, 'Test (ALL)')

    # Feature importance (normalized)
    imp_norm = pd.Series(model_all.feature_importances_, index=X_train_imp.columns).sort_values(ascending=False)
    selected_features = imp_norm.head(top_n).index.tolist()

    # 2) Retrain on TOP-N features
    model_top = train_xgb(X_train_imp[selected_features], y_train, X_test_imp[selected_features], y_test)
    _, r2_train_top, rmse_train_top = evaluate_model(model_top, X_train_imp[selected_features], y_train, f'Train (TOP-{top_n})')
    _, r2_test_top, rmse_test_top = evaluate_model(model_top, X_test_imp[selected_features], y_test, f'Test (TOP-{top_n})')

    results = {
        'Model': MODEL_NAME,
        'Parameter': param_name,
        'TopN': int(top_n),
        'R2_Train_All': r2_train_all,
        'RMSE_Train_All': rmse_train_all,
        'R2_Test_All': r2_test_all,
        'RMSE_Test_All': rmse_test_all,
        'R2_Train_Top': r2_train_top,
        'RMSE_Train_Top': rmse_train_top,
        'R2_Test_Top': r2_test_top,
        'RMSE_Test_Top': rmse_test_top
    }

    artifacts = {
        'model_all': model_all,
        'model_top': model_top,
        'imputer': imputer,
        'selected_features': selected_features,
        'importance': imp_norm
    }
    return artifacts, pd.DataFrame([results])


## 7) Train + evaluate (3 targets)

In [ ]:
# ============================================================
# 7) Train + evaluate (3 targets)
# - Baseline: XGB on BASE_FEATURE_COLS
# - Improved: Feature-training XGB on FULL numeric features (importance -> top N -> retrain)
# ============================================================
TOP_N = 80  # tune: 40/80/120

# Baseline (small feature set)
art_TA_base, res_TA_base = run_pipeline_feature_training(X_base, y_TA, 'Total Alkalinity (BASE)', top_n=min(TOP_N, X_base.shape[1]))
art_EC_base, res_EC_base = run_pipeline_feature_training(X_base, y_EC, 'Electrical Conductance (BASE)', top_n=min(TOP_N, X_base.shape[1]))
art_DRP_base, res_DRP_base = run_pipeline_feature_training(X_base, y_DRP, 'Dissolved Reactive Phosphorus (BASE)', top_n=min(TOP_N, X_base.shape[1]))

# Feature-training (wide numeric feature set)
art_TA_full, res_TA_full = run_pipeline_feature_training(X_full, y_TA, 'Total Alkalinity (FULL)', top_n=TOP_N)
art_EC_full, res_EC_full = run_pipeline_feature_training(X_full, y_EC, 'Electrical Conductance (FULL)', top_n=TOP_N)
art_DRP_full, res_DRP_full = run_pipeline_feature_training(X_full, y_DRP, 'Dissolved Reactive Phosphorus (FULL)', top_n=TOP_N)

# Summary
results_summary = pd.concat([
    res_TA_base, res_EC_base, res_DRP_base,
    res_TA_full, res_EC_full, res_DRP_full
], ignore_index=True)
display(results_summary)

# Mean R2 across 3 targets (Test TOP-N) for the FULL feature-training run
full_only = results_summary[results_summary['Parameter'].str.contains('(FULL)', regex=False)]
print('\nMean R² (Test, TOP-N) across 3 targets [FULL]:', float(full_only['R2_Test_Top'].mean()))

# Keep final artifacts (FULL TOP-N) for submission
MODEL_ARTIFACTS = {
    'TA': art_TA_full,
    'EC': art_EC_full,
    'DRP': art_DRP_full
}


## 8) Optional: Create submission.csv using validation features
This mirrors the benchmark submission flow. Requires these files in the notebook Files panel:
- `submission_template.csv`
- `landsat_features_validation.csv`
- `terraclimate_features_validation.csv`

In [ ]:
# ============================================================
# 8) Create submission.csv using validation features (feature-trained models)
# Uses per-target selected features + imputer from training.
# ============================================================
test_file = read_csv_strict('submission_template.csv')
landsat_val_features = read_csv_strict('landsat_features_validation.csv')
terraclimate_val_df = read_csv_strict('terraclimate_features_validation.csv')

# Ensure NDMI/MNDWI are numeric
for col in ['NDMI', 'MNDWI']:
    if col in landsat_val_features.columns:
        landsat_val_features[col] = pd.to_numeric(landsat_val_features[col], errors='coerce')

# Build wide validation feature table
val_data_full = pd.concat([landsat_val_features, terraclimate_val_df], axis=1)
val_data_full = val_data_full.loc[:, ~val_data_full.columns.duplicated()].copy()
val_data_full = val_data_full.apply(pd.to_numeric, errors='coerce')

def predict_with_artifact(artifact, val_df: pd.DataFrame):
    model = artifact['model_top']
    imputer = artifact['imputer']
    feats = artifact['selected_features']
    # Align columns; missing cols become NaN and will be imputed
    Xv = val_df.reindex(columns=feats)
    Xv_imp = pd.DataFrame(imputer.transform(Xv), columns=feats, index=val_df.index)
    return model.predict(Xv_imp)

pred_TA = predict_with_artifact(MODEL_ARTIFACTS['TA'], val_data_full)
pred_EC = predict_with_artifact(MODEL_ARTIFACTS['EC'], val_data_full)
pred_DRP = predict_with_artifact(MODEL_ARTIFACTS['DRP'], val_data_full)

submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'].values,
    'Latitude': test_file['Latitude'].values,
    'Sample Date': test_file['Sample Date'].values,
    'Total Alkalinity': pred_TA,
    'Electrical Conductance': pred_EC,
    'Dissolved Reactive Phosphorus': pred_DRP
})
display(submission_df.head())
submission_df.to_csv('/tmp/xg_boost_submission.csv', index=False)
print('Wrote /tmp/xg_boost_submission.csv')


### Optional: Upload submission file in Snowflake
Uncomment and run if you want to PUT the file into the notebook stage (same pattern as the benchmark notebook).

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql(f"""
PUT file:///tmp/xg_boost_submission.csv
'snow://workspace/USER$.PUBLIC.\"snowflake-challenge\"/versions/live/submission'
AUTO_COMPRESS=FALSE
OVERWRITE=TRUE
""").collect()
print('File saved! Refresh the browser to see it in the sidebar')